# Rerun of Top 10 Configurations (adult/pen-based)

## Imports

In [ ]:
import time
import json
import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix
from pathlib import Path

from Parser import Parser
from IBL import IBL
from processing_types import (
    NormalizationStrategy, EncodingStrategy,
    MissingValuesNumericStrategy, MissingValuesCategoricalStrategy, RetentionPolicy
)

BASE = "../datasetsCBR/datasetsCBR"
NUM_SPLITS = 10
ENCODING_FOR_METRIC ={
        "euclidean": EncodingStrategy.ONE_HOT_ENCODE,
        "cosine":    EncodingStrategy.ONE_HOT_ENCODE,
        "heom": EncodingStrategy.LABEL_ENCODE, 
}

DIR = Path("./results")
ADULT_PATH = DIR / "shortlist_for_rerun_adult.csv"
PENBASED_PATH = DIR / "shortlist_for_rerun_pen-based.csv"

  Using cached sklearn_relief-1.0.0b2-py3-none-any.whl.metadata (728 bytes)
Using cached sklearn_relief-1.0.0b2-py3-none-any.whl (8.7 kB)


## Runner

In [3]:
def cm_to_json(cm: np.ndarray, labels: list | None = None) -> str:
    d = {"labels": labels if labels is not None else list(range(cm.shape[0])),
         "matrix": cm.astype(int).tolist()}
    return json.dumps(d)

def run_suite(
    dataset_name: str,
    k: int,
    metric: str,
    retention: str,
    vote: str,
    out_csv: Path
):
    enc_strategy = ENCODING_FOR_METRIC[metric]

    parser = Parser(
        base_path=BASE,
        dataset_name=dataset_name,
        normalization_strategy=NormalizationStrategy.MEAN_NORMALIZE,
        encoding_strategy=enc_strategy, 
        missing_values_numeric_strategy=MissingValuesNumericStrategy.MEDIAN,
        missing_values_categorical_strategy=MissingValuesCategoricalStrategy.MODE,
        num_splits=NUM_SPLITS,
    )
    types = parser.get_types()

    splits = [parser.get_split(fold) for fold in range(NUM_SPLITS)]

    all_labels = set()
    for tr, te in splits:
        all_labels.update(np.unique(tr.iloc[:, -1]))
        all_labels.update(np.unique(te.iloc[:, -1]))
    labels = np.array(sorted(all_labels))


    rows = []
    for fold_id, (train_matrix, test_matrix) in enumerate(splits):
       
        ibl = IBL()

        # Fit + predict
        t0 = time.perf_counter()
        ibl.fit(train_matrix)
        t1 = time.perf_counter()
        preds = ibl.run(
            test_matrix,
            k=k,
            metric=metric,
            vote=vote,
            retention_policy=retention,
            types=types
        )
        t2 = time.perf_counter()

        # Times
        fit_time    = t1 - t0
        predict_time= t2 - t1
        total_time  = t2 - t0

        # Metrics
        y_true = test_matrix.iloc[:, -1].to_numpy()
        y_pred = np.asarray(preds)

        acc = accuracy_score(y_true, y_pred)

        pM, rM, fM, _ = precision_recall_fscore_support(
            y_true, y_pred, average="macro", zero_division=0
        )
        pW, rW, fW, _ = precision_recall_fscore_support(
            y_true, y_pred, average="weighted", zero_division=0
        )

        cm_fold = confusion_matrix(y_true, y_pred, labels=labels).astype(int)

        row = {
            "dataset": dataset_name,
            "metric": metric,
            "k": int(k),
            "vote": vote,
            "retention": retention,

            "fold_id": fold_id,
            "num_folds": NUM_SPLITS,
            "n_train": len(train_matrix),
            "n_test":  len(test_matrix),

            "fit_time_s": fit_time,
            "predict_time_s": predict_time,
            "total_time_s": total_time,

            "accuracy": acc,
            "precision_macro": pM,
            "recall_macro":    rM,
            "f1_macro":        fM,

            "precision_weighted": pW,
            "recall_weighted":    rW,
            "f1_weighted":        fW,

            "confusion_matrix_json": cm_to_json(cm_fold, labels=labels.tolist()),
        }
        rows.append(row)

    out_csv.parent.mkdir(parents=True, exist_ok=True)
    df_rows = pd.DataFrame(rows)

    write_header = not out_csv.exists()
    df_rows.to_csv(out_csv, mode="a", header=write_header, index=False)

    return df_rows


## Adult Rerun

In [ ]:
df_cfg = pd.read_csv(ADULT_PATH)

configs = [
    {
        "dataset":   row["dataset"],
        "metric":    row["metric"],     # distance metric
        "k":         int(row["k"]),
        "vote":      row["vote"],
        "retention": row["retention"],
    }
    for _, row in df_cfg.iterrows()
]

RET_MAP  = {
    "RetentionPolicy.ALWAYS_RETAIN": RetentionPolicy.ALWAYS_RETAIN,
    "RetentionPolicy.NEVER_RETAIN": RetentionPolicy.NEVER_RETAIN,
    "RetentionPolicy.DIFFERENT_CLASS_RETENTION": RetentionPolicy.DIFFERENT_CLASS_RETENTION,
    "RetentionPolicy.DD_RETENTION": RetentionPolicy.DD_RETENTION,
}

out_csv = DIR / "adult_rerun_results.csv"
for cfg in configs:
    print(f"dataset={cfg['dataset']}, metric={cfg['metric']}, k={cfg['k']}, vote={cfg['vote']}, retention={cfg['retention']}")
    run_suite(cfg['dataset'], k=cfg['k'], metric=cfg['metric'], retention=RET_MAP[cfg['retention']], vote=cfg['vote'], out_csv=out_csv)

dataset=adult, metric=cosine, k=7, vote=borda, retention=RetentionPolicy.ALWAYS_RETAIN
Preallocating matrix of shape (48842, 108)
Total time for all instances: 50.33s
Final training set size: (48842, 108)
Preallocating matrix of shape (48842, 108)
Total time for all instances: 50.37s
Final training set size: (48842, 108)
Preallocating matrix of shape (48842, 108)
Total time for all instances: 50.33s
Final training set size: (48842, 108)
Preallocating matrix of shape (48842, 108)
Total time for all instances: 50.28s
Final training set size: (48842, 108)
Preallocating matrix of shape (48842, 108)
Total time for all instances: 50.83s
Final training set size: (48842, 108)
Preallocating matrix of shape (48842, 108)
Total time for all instances: 50.60s
Final training set size: (48842, 108)
Preallocating matrix of shape (48842, 108)
Total time for all instances: 50.44s
Final training set size: (48842, 108)
Preallocating matrix of shape (48842, 108)
Total time for all instances: 50.78s
Final t

In [ ]:
df_cfg = pd.read_csv(PENBASED_PATH)

configs = [
    {
        "dataset":   row["dataset"],
        "metric":    row["metric"],     # distance metric
        "k":         int(row["k"]),
        "vote":      row["vote"],
        "retention": row["retention"],
    }
    for _, row in df_cfg.iterrows()
]

RET_MAP  = {
    "RetentionPolicy.ALWAYS_RETAIN": RetentionPolicy.ALWAYS_RETAIN,
    "RetentionPolicy.NEVER_RETAIN": RetentionPolicy.NEVER_RETAIN,
    "RetentionPolicy.DIFFERENT_CLASS_RETENTION": RetentionPolicy.DIFFERENT_CLASS_RETENTION,
    "RetentionPolicy.DD_RETENTION": RetentionPolicy.DD_RETENTION,
}

out_csv = DIR / "pen-based_rerun_results.csv"
for cfg in configs:
    print(f"dataset={cfg['dataset']}, metric={cfg['metric']}, k={cfg['k']}, vote={cfg['vote']}, retention={cfg['retention']}")
    run_suite(cfg['dataset'], k=cfg['k'], metric=cfg['metric'], retention=RET_MAP[cfg['retention']], vote=cfg['vote'], out_csv=out_csv)

dataset=pen-based, metric=cosine, k=5, vote=borda, retention=RetentionPolicy.ALWAYS_RETAIN
Preallocating matrix of shape (10992, 16)
Total time for all instances: 0.23s
Final training set size: (10992, 16)
Preallocating matrix of shape (10992, 16)
Total time for all instances: 0.23s
Final training set size: (10992, 16)
Preallocating matrix of shape (10992, 16)
Total time for all instances: 0.23s
Final training set size: (10992, 16)
Preallocating matrix of shape (10992, 16)
Total time for all instances: 0.23s
Final training set size: (10992, 16)
Preallocating matrix of shape (10992, 16)
Total time for all instances: 0.23s
Final training set size: (10992, 16)
Preallocating matrix of shape (10992, 16)
Total time for all instances: 0.23s
Final training set size: (10992, 16)
Preallocating matrix of shape (10992, 16)
Total time for all instances: 0.23s
Final training set size: (10992, 16)
Preallocating matrix of shape (10992, 16)
Total time for all instances: 0.23s
Final training set size: (

In [4]:
from itertools import product

out_csv = DIR / "adult_rerun_results.csv"

K_LIST      = [3, 5, 7]
METRICS     = ["cosine", "euclidean", "heom"]
VOTES       = ["borda", "modified_plurality"]
RETENTIONS  = [
    RetentionPolicy.ALWAYS_RETAIN,
    RetentionPolicy.NEVER_RETAIN,
    RetentionPolicy.DIFFERENT_CLASS_RETENTION,
    RetentionPolicy.DD_RETENTION,
]

for metric, k, vote, retention in product(METRICS, K_LIST, VOTES, RETENTIONS):
    print(f"dataset=adult, metric={metric}, k={k}, vote={vote}, retention={retention}")
    run_suite(
        dataset_name="adult",
        k=k,
        metric=metric,
        vote=vote,
        retention=retention,
        out_csv=out_csv
    )


dataset=adult, metric=cosine, k=3, vote=borda, retention=RetentionPolicy.ALWAYS_RETAIN
Preallocating matrix of shape (48842, 108)
Total time for all instances: 52.65s
Final training set size: (48842, 108)
Preallocating matrix of shape (48842, 108)
Total time for all instances: 51.46s
Final training set size: (48842, 108)
Preallocating matrix of shape (48842, 108)
Total time for all instances: 51.20s
Final training set size: (48842, 108)
Preallocating matrix of shape (48842, 108)
Total time for all instances: 50.89s
Final training set size: (48842, 108)
Preallocating matrix of shape (48842, 108)
Total time for all instances: 50.62s
Final training set size: (48842, 108)
Preallocating matrix of shape (48842, 108)
Total time for all instances: 50.57s
Final training set size: (48842, 108)
Preallocating matrix of shape (48842, 108)
Total time for all instances: 50.58s
Final training set size: (48842, 108)
Preallocating matrix of shape (48842, 108)
Total time for all instances: 50.72s
Final t

In [5]:
from itertools import product
dataset_name = "pen-based"
out_csv = DIR / "pen-based_rerun_results.csv"

K_LIST      = [3, 5, 7]
METRICS     = ["cosine", "euclidean", "heom"]
VOTES       = ["borda", "modified_plurality"]
RETENTIONS  = [
    RetentionPolicy.ALWAYS_RETAIN,
    RetentionPolicy.NEVER_RETAIN,
    RetentionPolicy.DIFFERENT_CLASS_RETENTION,
    RetentionPolicy.DD_RETENTION,
]

for metric, k, vote, retention in product(METRICS, K_LIST, VOTES, RETENTIONS):
    print(f"dataset={dataset_name}, metric={metric}, k={k}, vote={vote}, retention={retention}")
    run_suite(
        dataset_name=dataset_name,
        k=k,
        metric=metric,
        vote=vote,
        retention=retention,
        out_csv=out_csv
    )


dataset=pen-based, metric=cosine, k=3, vote=borda, retention=RetentionPolicy.ALWAYS_RETAIN
Preallocating matrix of shape (10992, 16)
Total time for all instances: 0.23s
Final training set size: (10992, 16)
Preallocating matrix of shape (10992, 16)
Total time for all instances: 0.23s
Final training set size: (10992, 16)
Preallocating matrix of shape (10992, 16)
Total time for all instances: 0.23s
Final training set size: (10992, 16)
Preallocating matrix of shape (10992, 16)
Total time for all instances: 0.23s
Final training set size: (10992, 16)
Preallocating matrix of shape (10992, 16)
Total time for all instances: 0.23s
Final training set size: (10992, 16)
Preallocating matrix of shape (10992, 16)
Total time for all instances: 0.23s
Final training set size: (10992, 16)
Preallocating matrix of shape (10992, 16)
Total time for all instances: 0.23s
Final training set size: (10992, 16)
Preallocating matrix of shape (10992, 16)
Total time for all instances: 0.23s
Final training set size: (